# NB09 — must-fix re-runs, shard 3 (CPU)

Jobs: `ablate` P3xhybrid; `refit` P6,P7. Confirmation runs use fresh frozen folds (seeds 5–9); refits train the GCN on train+val for the early-stopped epoch count of the matching main-grid run; classical baselines are refit on the GCN's training subjects only.

In [ ]:
REPO = 'https://github.com/Devguru-codes/Major_project_IIITN.git'
BRANCH = 'feat/implementation'
WORK = '/kaggle/working'
SRC = '/tmp/eegrep-src'   # code checkout stays out of the saved output

import subprocess, sys, os, json, platform, shutil

def sh(cmd, log=None):
    print('$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout[-20000:], r.stderr[-5000:], sep='\n', flush=True)
    if log:
        open(log, 'w').write(r.stdout + '\n' + r.stderr)
    if r.returncode != 0:
        raise RuntimeError(f'command failed ({r.returncode}): {cmd}')
    return r.stdout

shutil.rmtree(SRC, ignore_errors=True)
sh(f'git clone --depth 1 -b {BRANCH} {REPO} {SRC}')
SHA = sh(f'git -C {SRC} rev-parse --short HEAD').strip()
sh(f'pip install -q -e "{SRC}[dev]"', log=f'{WORK}/pip_install.log')
PY = f'cd {SRC} && {sys.executable} -m eegrep'
print('git sha', SHA)

In [ ]:
import glob
CACHE = [d for d in glob.glob('/kaggle/input/**/cache', recursive=True) if os.path.exists(f'{d}/index.npz')][0]
PART = sorted(glob.glob('/kaggle/input/**/participants.tsv', recursive=True))[0]
FOLDS = f'{SRC}/splits/folds_ds004504.json'
print(CACHE, PART, FOLDS)

In [ ]:
sh(f'{PY} ablate --cache {CACHE} --folds {SRC}/splits/folds_ds004504_confirm.json --participants {PART} --cells P3xhybrid --which all --out {WORK}/ablations_confirm.jsonl --max-hours 6', log=f'{WORK}/ablate_0.log')

In [ ]:
sh(f'{PY} grid --cache {CACHE} --folds {FOLDS} --pipelines P6,P7 --edges spatial,functional,hybrid --tag refit --refit-from {SRC}/reports/grid/runs_s*.jsonl --out {WORK}/refit_runs.jsonl --max-hours 5', log=f'{WORK}/refit_1.log')

In [ ]:
import torch
sh(f'{sys.executable} -m pip freeze', log=f'{WORK}/pip_freeze.txt')
env = {'git_sha': SHA, 'python': platform.python_version(), 'torch': torch.__version__,
       'cuda': torch.cuda.is_available(), 'cpu_count': os.cpu_count(),
       'kaggle_image': os.environ.get('KAGGLE_DOCKER_IMAGE', 'unknown')}
json.dump(env, open(f'{WORK}/environment.json', 'w'), indent=1)
print(env)